In [ ]:
# Exoplanet Detection with Machine Learning
# Baseline Pipeline on Kepler KOI Dataset

# 1. Imports

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
classification_report,
confusion_matrix,
roc_curve,
auc
)



In [ ]:
# 2. Load Dataset: Replace the path with your dataset location.

import pandas as pd
df = pd.read_csv("../data/raw/cumulative.csv")


In [ ]:
# 3. Quick Overview

# First 5 rows
df.head()

# Dataset shape
df.shape

# General information
df.info()

# Statistical summary
df.describe()



In [ ]:
# 4. Check Target Classes: koi_disposition is the target column.
# Possible values: CONFIRMED --- FALSE POSITIVE --- CANDIDATE
df["koi_disposition"].value_counts()

In [ ]:
# 5. Remove CANDIDATE Samples
# For baseline ML, we use only:CONFIRMED → 1 --- FALSE POSITIVE → 0
df = df[df["koi_disposition"] != "CANDIDATE"]


In [ ]:
# 6. Create Binary Labels
df["label"] = df["koi_disposition"].map({
"CONFIRMED": 1,
"FALSE POSITIVE": 0
})


In [ ]:
# 7. Class Distribution Visualization
sns.countplot(data=df, x="label")

plt.title("Class Distribution")
plt.xticks([0, 1], ["False Positive", "Confirmed"])

plt.show()


In [ ]:
# 8. Select Important Features: These are scientifically meaningful features for exoplanet detection.
features = [
"koi_period",
"koi_impact",
"koi_duration",
"koi_depth",
"koi_prad",
"koi_teq",
"koi_insol",
"koi_model_snr",
"koi_steff",
"koi_slogg",
"koi_srad",
"koi_kepmag"
]



In [ ]:
# 9. Create Feature Matrix and Target Vector
X = df[features]
y = df["label"]


In [ ]:
# 10. Handle Missing Values: We fill missing values using median.
X = X.fillna(X.median())



In [ ]:
# 11. Feature Distribution Visualization
#Histogram of Transit Depth
sns.histplot(data=df, x="koi_depth", hue="label", bins=50)

plt.title("Distribution of Transit Depth")
plt.show()


In [ ]:
#Histogram of Orbital Period
sns.histplot(data=df, x="koi_period", hue="label", bins=50)

plt.title("Distribution of Orbital Period")
plt.show()


In [ ]:
#Histogram of Planet Radius
sns.histplot(data=df, x="koi_prad", hue="label", bins=50)

plt.title("Distribution of Planet Radius")
plt.show()


In [ ]:
#Histogram of Signal-to-Noise Ratio
sns.histplot(data=df, x="koi_model_snr", hue="label", bins=50)

plt.title("Distribution of SNR")
plt.show()


In [ ]:
# 12. Correlation Matrix: This helps us understand relationships between features.
plt.figure(figsize=(12, 8))

corr_matrix = X.corr()

sns.heatmap(
corr_matrix,
cmap="coolwarm",
annot=True,
fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
# 13. Train/Test Split
#We use stratify=y to preserve class balance.
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)


In [ ]:
# 14. Feature Scaling
#Scaling is important for Logistic Regression.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# 15. Logistic Regression Model
#Train Model
log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_scaled, y_train)

#Predictions
y_pred_lr = log_reg.predict(X_test_scaled)

#Classification Report
print(classification_report(y_test, y_pred_lr))


In [ ]:
#Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)

sns.heatmap(
cm,
annot=True,
fmt="d",
cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")

plt.show()


In [ ]:
# 16. ROC Curve for Logistic Regression
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob_lr)

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Logistic Regression")
plt.legend()

plt.show()


In [ ]:
# 17. Random Forest Model
#Train Model
rf = RandomForestClassifier(
n_estimators=200,
random_state=42
)

rf.fit(X_train, y_train)

#Predictions
y_pred_rf = rf.predict(X_test)

#Classification Report
print(classification_report(y_test, y_pred_rf))

#Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(
cm_rf,
annot=True,
fmt="d",
cmap="Greens"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")

plt.show()


In [ ]:
#18. ROC Curve for Random Forest
y_prob_rf = rf.predict_proba(X_test)[:, 1]

fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_prob_rf)

roc_auc_rf = auc(fpr_rf, tpr_rf)

plt.plot(fpr_rf, tpr_rf, label=f"AUC = {roc_auc_rf:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Random Forest")
plt.legend()

plt.show()
